<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/iaa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inter-Annotator Agreement (IAA) — Krippendorff's α via BERTScore

Pipeline:
1. Pull `human_insights` rows from Supabase
2. Build (dashboard_id, chart_id, level) units
3. Compute BERTScore-based pairwise distances for all 3 annotator pairs
4. Compute Krippendorff's α overall, per level, and per dashboard

## Dependencies

In [1]:
!pip install -q bert-score supabase pandas numpy openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.8 MB/s eta 0:00:00


## STEP 1: Pull data from Supabase

In [2]:
import pandas as pd
import numpy as np
import re
from bert_score import score as bert_score
from supabase import create_client
from itertools import combinations
from google.colab import userdata

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
response = supabase.table("human_insights").select("*").execute()
df_raw = pd.DataFrame(response.data)

print("irr_flag unique values:", df_raw["irr_flag"].unique())
irr_df = df_raw[df_raw["irr_flag"] == True].reset_index(drop=True)
print(f"IRR rows: {len(irr_df)}")

irr_flag unique values: [False  True]
IRR rows: 10


## STEP 2: Parsing step

In [3]:
NA_PATTERN = re.compile(r"^(not applicable|n/a)$", re.IGNORECASE)

def normalize_value(val):
    """Post-parse normalization: catch any NA variants that slipped through."""
    if val is None:
        return None
    cleaned = str(val).strip()
    if cleaned == "" or NA_PATTERN.match(cleaned):
        return None
    return cleaned

def parse_charts(text):
    charts = []
    if not text or (isinstance(text, float) and np.isnan(text)):
        return charts

    blocks = re.split(r"(?=Chart\s+\d+\s*[:.])", text.strip())
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        header = lines[0].strip()
        match = re.match(r"Chart\s+(\d+)\s*[:.]\s*(.*)", header)
        if not match:
            continue
        chart_id = int(match.group(1))
        title    = match.group(2).strip()

        L2 = L3 = L4 = None
        for line in lines[1:]:
            line = line.strip()
            if line.startswith("L2:"):
                L2 = normalize_value(line[3:].strip())
            elif line.startswith("L3:"):
                L3 = normalize_value(line[3:].strip())
            elif line.startswith("L4:"):
                L4 = normalize_value(line[3:].strip())

        charts.append({
            "chart_id": chart_id,
            "title":    title,
            "L2":       L2,
            "L3":       L3,
            "L4":       L4,
        })
    return charts

In [4]:
ANNOTATOR_COLS = ["insight_part_1", "insight_part_2", "insight_part_3"]

records = []
for _, row in irr_df.iterrows():
    row_id      = row["id"]
    metadata_id = row["metadata_id"]

    # Parse each annotator's blob
    parsed = {col: {c["chart_id"]: c for c in parse_charts(row[col])}
              for col in ANNOTATOR_COLS}

    # All chart_ids seen across any annotator
    all_chart_ids = sorted(
        set().union(*[set(p.keys()) for p in parsed.values()])
    )

    for chart_id in all_chart_ids:
        # Get title from whichever annotator has this chart
        title = next(
            (parsed[col][chart_id]["title"]
             for col in ANNOTATOR_COLS
             if chart_id in parsed[col]),
            ""
        )
        for level in ["L2", "L3", "L4"]:
            records.append({
                "row_id":      row_id,
                "metadata_id": metadata_id,
                "chart_id":    chart_id,
                "title":       title,
                "level":       level,
                "unit_id":     f"{row_id}__chart{chart_id}__{level}",
                "annotator_1": parsed["insight_part_1"].get(chart_id, {}).get(level),
                "annotator_2": parsed["insight_part_2"].get(chart_id, {}).get(level),
                "annotator_3": parsed["insight_part_3"].get(chart_id, {}).get(level),
            })

long_df = pd.DataFrame(records)
print(f"\nTotal units: {len(long_df)}")
print(long_df.head(12).to_string())


Total units: 147
                                  row_id                           metadata_id  chart_id                        title level                                           unit_id                                                                                                                                                                                                                                                                                              annotator_1                                                                                                                                                                                                                                                                                                               annotator_2                                                                                                                                                                                                      

In [5]:
print("\nNOT APPLICABLE counts per level per annotator:")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}: "
          f"ann1={sub['annotator_1'].isna().sum()} | "
          f"ann2={sub['annotator_2'].isna().sum()} | "
          f"ann3={sub['annotator_3'].isna().sum()} | "
          f"total_units={len(sub)}")


NOT APPLICABLE counts per level per annotator:
  L2: ann1=0 | ann2=0 | ann3=4 | total_units=49
  L3: ann1=24 | ann2=3 | ann3=2 | total_units=49
  L4: ann1=0 | ann2=0 | ann3=1 | total_units=49


In [6]:
# Check chart count per dashboard per annotator
print("=== Chart count per dashboard per annotator ===\n")

for row_id in sorted(long_df["row_id"].unique()):
    sub = long_df[long_df["row_id"] == row_id]

    # Get unique charts seen per annotator
    ann1_charts = sub[sub["annotator_1"].notna()]["chart_id"].unique()
    ann2_charts = sub[sub["annotator_2"].notna()]["chart_id"].unique()
    ann3_charts = sub[sub["annotator_3"].notna()]["chart_id"].unique()

    total_charts = sub["chart_id"].nunique()
    metadata_id = sub["metadata_id"].iloc[0]

    print(f"Dashboard: {row_id[:8]}… (metadata: {metadata_id[:8]}…)")
    print(f"  Total charts parsed: {total_charts}")
    print(f"  ann1 charts with content: {sorted(ann1_charts)} ({len(ann1_charts)})")
    print(f"  ann2 charts with content: {sorted(ann2_charts)} ({len(ann2_charts)})")
    print(f"  ann3 charts with content: {sorted(ann3_charts)} ({len(ann3_charts)})")

    # Flag mismatches
    all_charts = set(sub["chart_id"].unique())
    for ann_label, ann_charts in [("ann1", ann1_charts), ("ann2", ann2_charts), ("ann3", ann3_charts)]:
        missing = all_charts - set(ann_charts)
        if missing:
            print(f"  ⚠️  {ann_label} missing charts: {sorted(missing)}")
    print()

=== Chart count per dashboard per annotator ===

Dashboard: 08006d53… (metadata: 27052e58…)
  Total charts parsed: 4
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)

Dashboard: 1120a289… (metadata: 4f4b551b…)
  Total charts parsed: 5
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(5)] (4)
  ⚠️  ann3 missing charts: [np.int64(4)]

Dashboard: 2e5b2881… (metadata: 932c11c0…)
  Total charts parsed: 6
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)] (6)
  ann2 charts with content: [np.int64(1), np.int64(2), np

## STEP 3: BERTScore-based distance function

`distance(a, b) = 1 - BERTScore_F1(a, b)`  
`NOT APPLICABLE` entries are treated as missing (`np.nan`).

In [7]:
import torch
import numpy as np
from bert_score import score as bert_score
from itertools import combinations

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_TYPE = "roberta-large"
print(f"Using device: {DEVICE}, model: {MODEL_TYPE}")

ANNOTATOR_COLS = ["annotator_1", "annotator_2", "annotator_3"]
ANNOTATOR_PAIRS = list(combinations(ANNOTATOR_COLS, 2))

def is_missing(text):
    if text is None:
        return True
    return str(text).strip().lower() in {"not applicable", "n/a", ""}

def bertscore_distance_batch(refs, hyps, model_type=MODEL_TYPE, device=DEVICE):
    """
    Compute BERTScore F1 for parallel lists of refs and hyps.
    Returns a numpy array of distances (1 - F1).
    Missing values return np.nan.
    """
    assert len(refs) == len(hyps)
    distances = np.full(len(refs), np.nan)

    valid_indices = [
        i for i, (r, h) in enumerate(zip(refs, hyps))
        if not is_missing(r) and not is_missing(h)
    ]

    if not valid_indices:
        return distances

    valid_refs = [str(refs[i]) for i in valid_indices]
    valid_hyps = [str(hyps[i]) for i in valid_indices]

    _, _, F1 = bert_score(
        valid_hyps, valid_refs,
        model_type=model_type,
        device=device,
        verbose=False
    )

    for idx, f1_val in zip(valid_indices, F1.numpy()):
        distances[idx] = 1.0 - f1_val

    return distances

# ============================================================
# Compute pairwise distances on long_df
# ============================================================
DIST_COLS = []
for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"dist_{suffix_a}{suffix_b}"
    DIST_COLS.append(pair_key)
    print(f"Computing {pair_key} ({col_a} vs {col_b}) ...")
    long_df[pair_key] = bertscore_distance_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist(),
    )

print("\nSample distances:")
print(long_df[["unit_id", "chart_id", "level"] + DIST_COLS].head(12).to_string(index=False))

Using device: cuda, model: roberta-large
Computing dist_12 (annotator_1 vs annotator_2) ...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Computing dist_23 (annotator_2 vs annotator_3) ...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Sample distances:
                                         unit_id  chart_id level  dist_12  dist_13  dist_23
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L2         1    L2 0.057711 0.075818 0.056742
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L3         1    L3      NaN      NaN      NaN
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L4         1    L4 0.120168 0.126460 0.118056
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L2         2    L2 0.089606      NaN      NaN
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L3         2    L3 0.124904 0.135269 0.116033
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L4         2    L4 0.122942 0.133093 0.100707
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart3__L2         3    L2 0.060297 0.073094 0.067503
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart3__L3         3    L3      NaN      NaN 0.116616
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart3__L4         3    L4 0.142960 0.145438 0.132202
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart4__L2         4   

## STEP 4: Krippendorff's alpha with custom BERTScore distance

$$\alpha = 1 - \frac{D_o}{D_e}$$

- **D_o** = mean observed disagreement (average pairwise distance within units, >=2 valid annotators)
- **D_e** = mean expected disagreement (average over all valid distances in the pool)

In [12]:
def compute_cross_item_distances_batched(sub_df, ann_col, model_type=MODEL_TYPE, device=DEVICE):
    annotations = sub_df[ann_col].tolist()
    n = len(annotations)

    refs_batch = []
    hyps_batch = []

    for i in range(n):
        for j in range(i + 1, n):
            a, b = annotations[i], annotations[j]
            if is_missing(a) or is_missing(b):
                continue
            refs_batch.append(str(a))
            hyps_batch.append(str(b))

    if not refs_batch:
        return []

    _, _, F1 = bert_score(
        hyps_batch, refs_batch,
        model_type=model_type,
        device=device,
        verbose=False
    )
    return (1.0 - F1.numpy()).tolist()


def krippendorff_alpha_bertscore_correct(sub_df, dist_cols=DIST_COLS):
    unit_mean_dist = sub_df[dist_cols].mean(axis=1, skipna=True)
    valid_units    = unit_mean_dist.notna()
    n_valid        = valid_units.sum()

    if n_valid < 2:
        return np.nan, np.nan, np.nan, n_valid

    D_o = unit_mean_dist[valid_units].mean()

    cross_item_distances = []
    for ann_col in ["annotator_1", "annotator_2", "annotator_3"]:
        print(f"  [{ann_col}] computing cross-item distances "
              f"({sub_df[ann_col].notna().sum()} valid annotations)...")
        cross_item_distances.extend(
            compute_cross_item_distances_batched(sub_df, ann_col)
        )

    if not cross_item_distances:
        return np.nan, np.nan, np.nan, n_valid

    D_e = np.mean(cross_item_distances)
    if D_e == 0:
        return np.nan, np.nan, np.nan, n_valid

    alpha = 1.0 - (D_o / D_e)
    return alpha, D_o, D_e, n_valid

## STEP 6: Overall alpha (all dashboards combined)

In [14]:
print("=" * 60)
print(f"Krippendorff's α — BERTScore ({MODEL_TYPE})")
print("=" * 60)

print("\n[Overall]")
alpha_all, Do_all, De_all, n_all = krippendorff_alpha_bertscore_correct(long_df)
print(f"  α={alpha_all:.4f}  D_o={Do_all:.4f}  D_e={De_all:.4f}  n_valid={n_all}")

Krippendorff's α — BERTScore (roberta-large)

[Overall]
  [annotator_1] computing cross-item distances (123 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [annotator_2] computing cross-item distances (144 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [annotator_3] computing cross-item distances (140 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  α=0.2348  D_o=0.1160  D_e=0.1515  n_valid=144


## STEP 7: Alpha per insight level (L2 / L3 / L4)

In [15]:
print("\n[By Semantic Level]")
level_results = []
for level in ["L2", "L3", "L4"]:
    print(f"\n  Level {level}:")
    sub = long_df[long_df["level"] == level]
    alpha, D_o, D_e, n_valid = krippendorff_alpha_bertscore_correct(sub)
    level_results.append({
        "Level":         level,
        "Alpha":         round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "D_o":           round(D_o, 4)   if not np.isnan(D_o)   else "N/A",
        "D_e":           round(D_e, 4)   if not np.isnan(D_e)   else "N/A",
        "N_units_valid": n_valid,
        "N_units_total": len(sub),
    })
    print(f"  → α={alpha:.4f}  D_o={D_o:.4f}  D_e={D_e:.4f}  "
          f"valid={n_valid}/{len(sub)}")

level_df = pd.DataFrame(level_results)


[By Semantic Level]

  Level L2:
  [annotator_1] computing cross-item distances (49 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [annotator_2] computing cross-item distances (49 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [annotator_3] computing cross-item distances (48 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  → α=0.0381  D_o=0.1284  D_e=0.1334  valid=49/49


## STEP 8: Alpha per dashboard

In [16]:
print("\n[By Dashboard]")
dash_results = []
for row_id in sorted(long_df["row_id"].unique()):
    print(f"\n  Dashboard {row_id[:8]}…:")
    sub = long_df[long_df["row_id"] == row_id]
    alpha, D_o, D_e, n_valid = krippendorff_alpha_bertscore_correct(sub)
    dash_results.append({
        "row_id":        row_id,
        "Alpha":         round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "D_o":           round(D_o, 4)   if not np.isnan(D_o)   else "N/A",
        "D_e":           round(D_e, 4)   if not np.isnan(D_e)   else "N/A",
        "N_units_valid": n_valid,
        "N_units_total": len(sub),
    })
    print(f"  → α={alpha:.4f}  valid={n_valid}/{len(sub)}")

dash_df = pd.DataFrame(dash_results)


[By Dashboard]

  Dashboard 08006d53…:
  [annotator_1] computing cross-item distances (9 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [annotator_2] computing cross-item distances (12 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [annotator_3] computing cross-item distances (11 valid annotations)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  → α=0.2963  valid=11/12


## STEP 9: Summary printout

In [17]:
print("\n[Level Summary]")
print(level_df.to_string(index=False))

print("\n[Dashboard Summary]")
print(dash_df.to_string(index=False))


[Level Summary]
Level  Alpha    D_o    D_e  N_units_valid  N_units_total
   L2 0.3357 0.0938 0.1413             49             49
   L3 0.1039 0.1263 0.1409             46             49
   L4 0.0381 0.1284 0.1334             49             49

[Dashboard Summary]
                              row_id  Alpha    D_o    D_e  N_units_valid  N_units_total
08006d53-f01b-4eb6-a517-8825c184d732 0.2613 0.1054 0.1427             12             12
1120a289-6f8e-4498-91d8-59bb0b7bb49d 0.2718 0.1099 0.1509             14             15
2e5b2881-e3e3-421f-a3b9-098e326303a3 0.1989 0.1154 0.1441             18             18
511bca34-e364-49d6-8cf9-263c3426db13 0.1931 0.1295 0.1604              9              9
723340b5-2604-49f0-963a-3699aef72048 0.1405 0.1187 0.1381             12             12
b164ac02-5f8b-4b10-ad35-afa018951888 0.1760 0.1242 0.1508             18             18
cfe29540-1433-470e-9dc8-612e018bd8b6 0.2171 0.1130 0.1443             21             21
dc09d834-178b-48e2-990d-944644

## STEP 10: Moving on to BERTScore

In [18]:
F1_COLS = {
    "dist_12": "F1_ann1_ann2",
    "dist_13": "F1_ann1_ann3",
    "dist_23": "F1_ann2_ann3",
}

for dist_col, f1_col in F1_COLS.items():
    long_df[f1_col] = 1.0 - long_df[dist_col]  # NaN stays NaN

F1_VALUE_COLS = list(F1_COLS.values())

print("--- Mean Pairwise BERTScore F1 by Level ---")
iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}
    for f1_col in F1_VALUE_COLS:
        mean_f1 = sub[f1_col].mean(skipna=True)
        n_valid  = sub[f1_col].notna().sum()
        row[f1_col]               = round(mean_f1, 4)
        row[f1_col + "_n_valid"]  = n_valid
    row["Mean_F1_overall"] = round(
        sub[F1_VALUE_COLS].values.flatten()[
            ~np.isnan(sub[F1_VALUE_COLS].values.flatten())
        ].mean(), 4
    )
    iaa_results.append(row)
    print(f"  {level}: "
          f"ann1-ann2={row['F1_ann1_ann2']:.4f} (n={row['F1_ann1_ann2_n_valid']})  "
          f"ann1-ann3={row['F1_ann1_ann3']:.4f} (n={row['F1_ann1_ann3_n_valid']})  "
          f"ann2-ann3={row['F1_ann2_ann3']:.4f} (n={row['F1_ann2_ann3_n_valid']})  "
          f"mean={row['Mean_F1_overall']:.4f}")

iaa_df = pd.DataFrame(iaa_results)

--- Mean Pairwise BERTScore F1 by Level ---
  L2: ann1-ann2=0.9088 (n=49)  ann1-ann3=0.9065 (n=45)  ann2-ann3=0.9054 (n=45)  mean=0.9070
  L3: ann1-ann2=0.8781 (n=25)  ann1-ann3=0.8721 (n=25)  ann2-ann3=0.8747 (n=46)  mean=0.8749
  L4: ann1-ann2=0.8683 (n=49)  ann1-ann3=0.8661 (n=48)  ann2-ann3=0.8805 (n=48)  mean=0.8716


### Qualitative analysis: lowest scores

In [20]:
print("=== Lowest Pairwise BERTScore F1 Units ===\n")

N_BOTTOM = 5  # how many lowest to inspect per level

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()

    # Mean F1 across available pairs per unit
    sub["mean_F1"] = sub[F1_VALUE_COLS].mean(axis=1, skipna=True)

    # Only units with at least one valid pair
    sub_valid = sub[sub["mean_F1"].notna()].sort_values("mean_F1")

    print(f"{'='*70}")
    print(f"LEVEL {level} — Bottom {N_BOTTOM} units by mean pairwise F1")
    print(f"{'='*70}")

    for _, row in sub_valid.head(N_BOTTOM).iterrows():
        print(f"\nunit : {row['unit_id']}")
        print(f"chart: {row['chart_id']} — {row['title']}")
        print(f"F1   : ann1-ann2={row['F1_ann1_ann2']:.4f}  "
              f"ann1-ann3={row['F1_ann1_ann3']:.4f}  "
              f"ann2-ann3={row['F1_ann2_ann3']:.4f}  "
              f"mean={row['mean_F1']:.4f}")
        print(f"ann1 : {row['annotator_1']}")
        print(f"ann2 : {row['annotator_2']}")
        print(f"ann3 : {row['annotator_3']}")
        print()

=== Lowest Pairwise BERTScore F1 Units ===

LEVEL L2 — Bottom 5 units by mean pairwise F1

unit : 511bca34-e364-49d6-8cf9-263c3426db13__chart2__L2
chart: 2 — Profit Margin by State
F1   : ann1-ann2=0.8454  ann1-ann3=nan  ann2-ann3=nan  mean=0.8454
ann1 : The majority of states show a positive margin of green encoded in 2021, compared to 2020.
ann2 : AK, WY, HI, and ME had N/A value.
ann3 : None


unit : b164ac02-5f8b-4b10-ad35-afa018951888__chart2__L2
chart: 2 — 2023 | Sales vs Targets
F1   : ann1-ann2=0.8587  ann1-ann3=0.8687  ann2-ann3=0.8413  mean=0.8562
ann1 : Sales started at around £40K in January and ended the year at around £80K, but it did not reach the target.
ann2 : In 2023, the sales vs target, J is 21.7K, F is -7.3K, and M is 3.2K.
ann3 : January, June, August, October, and November were months with sales above target.


unit : dc09d834-178b-48e2-990d-94464475d0fa__chart5__L2
chart: 5 — Monthly Orders Details
F1   : ann1-ann2=0.8459  ann1-ann3=0.8893  ann2-ann3=0.8679  mea

### Qualitative analysis: highest scores

In [21]:
print("\n=== Highest Pairwise BERTScore F1 Units ===\n")

N_TOP = 3

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()
    sub["mean_F1"] = sub[F1_VALUE_COLS].mean(axis=1, skipna=True)
    sub_valid = sub[sub["mean_F1"].notna()].sort_values("mean_F1", ascending=False)

    print(f"{'='*70}")
    print(f"LEVEL {level} — Top {N_TOP} units by mean pairwise F1")
    print(f"{'='*70}")

    for _, row in sub_valid.head(N_TOP).iterrows():
        print(f"\nunit : {row['unit_id']}")
        print(f"chart: {row['chart_id']} — {row['title']}")
        print(f"F1   : ann1-ann2={row['F1_ann1_ann2']:.4f}  "
              f"ann1-ann3={row['F1_ann1_ann3']:.4f}  "
              f"ann2-ann3={row['F1_ann2_ann3']:.4f}  "
              f"mean={row['mean_F1']:.4f}")
        print(f"ann1 : {row['annotator_1']}")
        print(f"ann2 : {row['annotator_2']}")
        print(f"ann3 : {row['annotator_3']}")
        print()


=== Highest Pairwise BERTScore F1 Units ===

LEVEL L2 — Top 3 units by mean pairwise F1

unit : 1120a289-6f8e-4498-91d8-59bb0b7bb49d__chart1__L2
chart: 1 — Scoreboard Overview
F1   : ann1-ann2=0.9727  ann1-ann3=0.9454  ann2-ann3=0.9424  mean=0.9535
ann1 : Sales are 733,215, profit is 93,439, and the number of orders is 1,687.
ann2 : Sales are 733,215, profit is 93.439, and orders are 1,687.
ann3 : Sales value is 733,215, Profit value is 93,439, and Orders values is 1,687


unit : 08006d53-f01b-4eb6-a517-8825c184d732__chart1__L2
chart: 1 — Scoreboard & Barchart
F1   : ann1-ann2=0.9709  ann1-ann3=0.9435  ann2-ann3=0.9391  mean=0.9511
ann1 : Total sales are $745.6K, total orders are $3.4K, and sales are $5.4K above the target.
ann2 : Total sales are $745.6K, total number of orders is 3.4K, and the sales are $5.4 above target.
ann3 : Total sales is 745.6K, Total Orders is 3.4K and Sales is above target with 5.4K in value


unit : cfe29540-1433-470e-9dc8-612e018bd8b6__chart1__L2
chart: 1 —

### Distribution summary per level

In [22]:
print("\n=== F1 Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()
    sub["mean_F1"] = sub[F1_VALUE_COLS].mean(axis=1, skipna=True)
    vals = sub["mean_F1"].dropna()
    print(f"  {level}: min={vals.min():.4f}  "
          f"Q1={vals.quantile(0.25):.4f}  "
          f"median={vals.quantile(0.50):.4f}  "
          f"Q3={vals.quantile(0.75):.4f}  "
          f"max={vals.max():.4f}")


=== F1 Distribution Summary per Level ===

  L2: min=0.8454  Q1=0.8902  median=0.9033  Q3=0.9258  max=0.9535
  L3: min=0.8312  Q1=0.8635  median=0.8752  Q3=0.8847  max=0.9142
  L4: min=0.8510  Q1=0.8657  median=0.8727  Q3=0.8785  max=0.8893


## STEP 11: ROUGE trial

In [24]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6aaf1115113eee75d610457d58765b62b5908310593faa639151451f51defe4d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [25]:
from rouge_score import rouge_scorer
import numpy as np

# ============================================================
# ROUGE scorer setup
# We use ROUGE-1, ROUGE-2, and ROUGE-L
# ============================================================
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

# ============================================================
# Pairwise ROUGE F1 between two text lists
# Returns dict of {metric: numpy array of scores}
# Missing values return np.nan
# ============================================================
def rouge_pairwise_batch(refs, hyps):
    assert len(refs) == len(hyps)

    results = {
        "rouge1": np.full(len(refs), np.nan),
        "rouge2": np.full(len(refs), np.nan),
        "rougeL": np.full(len(refs), np.nan),
    }

    for i, (r, h) in enumerate(zip(refs, hyps)):
        if is_missing(r) or is_missing(h):
            continue
        scores = scorer.score(str(r), str(h))
        results["rouge1"][i] = scores["rouge1"].fmeasure
        results["rouge2"][i] = scores["rouge2"].fmeasure
        results["rougeL"][i] = scores["rougeL"].fmeasure

    return results

# ============================================================
# Compute pairwise ROUGE for all annotator pairs
# ============================================================
ROUGE_METRICS = ["rouge1", "rouge2", "rougeL"]
ROUGE_PAIR_COLS = {}  # maps (pair_key, metric) -> column name

for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"{suffix_a}{suffix_b}"

    print(f"Computing ROUGE for ann{suffix_a} vs ann{suffix_b} ...")
    results = rouge_pairwise_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist()
    )

    for metric in ROUGE_METRICS:
        col_name = f"rouge_{metric}_{pair_key}"
        long_df[col_name] = results[metric]
        ROUGE_PAIR_COLS[(pair_key, metric)] = col_name

print("\nDone. Sample:")
sample_cols = ["unit_id", "level"] + list(ROUGE_PAIR_COLS.values())[:6]
print(long_df[sample_cols].head(6).to_string(index=False))

# ============================================================
# Mean pairwise ROUGE F1 per level
# ============================================================
print("\n=== Mean Pairwise ROUGE F1 by Level ===\n")

rouge_iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}

    for metric in ROUGE_METRICS:
        pair_scores = []
        pair_details = []

        for (col_a, col_b) in ANNOTATOR_PAIRS:
            suffix_a = col_a.split("_")[-1]
            suffix_b = col_b.split("_")[-1]
            pair_key = f"{suffix_a}{suffix_b}"
            col_name = ROUGE_PAIR_COLS[(pair_key, metric)]

            mean_score = sub[col_name].mean(skipna=True)
            n_valid    = sub[col_name].notna().sum()
            pair_scores.append(mean_score)
            pair_details.append(f"ann{suffix_a}-ann{suffix_b}={mean_score:.4f}(n={n_valid})")

            row[f"{metric}_ann{suffix_a}_ann{suffix_b}"] = round(mean_score, 4)
            row[f"{metric}_ann{suffix_a}_ann{suffix_b}_n"] = n_valid

        row[f"{metric}_mean"] = round(np.nanmean(pair_scores), 4)
        print(f"  {level} {metric.upper():8s}: "
              f"{' | '.join(pair_details)} | mean={row[f'{metric}_mean']:.4f}")

    rouge_iaa_results.append(row)
    print()

rouge_iaa_df = pd.DataFrame(rouge_iaa_results)

# ============================================================
# Distribution summary per level per metric
# ============================================================
print("\n=== ROUGE Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}:")
    for metric in ROUGE_METRICS:
        # Pool all pair scores for this level+metric
        pair_cols = [ROUGE_PAIR_COLS[(f"{a.split('_')[-1]}{b.split('_')[-1]}", metric)]
                     for (a, b) in ANNOTATOR_PAIRS]
        vals = sub[pair_cols].values.flatten()
        vals = vals[~np.isnan(vals)]
        print(f"    {metric.upper():8s}: min={vals.min():.4f}  "
              f"Q1={vals.quantile(0.25) if hasattr(vals, 'quantile') else np.percentile(vals, 25):.4f}  "
              f"median={np.median(vals):.4f}  "
              f"Q3={np.percentile(vals, 75):.4f}  "
              f"max={vals.max():.4f}")
    print()

Computing ROUGE for ann1 vs ann2 ...
Computing ROUGE for ann1 vs ann3 ...
Computing ROUGE for ann2 vs ann3 ...

Done. Sample:
                                         unit_id level  rouge_rouge1_12  rouge_rouge2_12  rouge_rougeL_12  rouge_rouge1_13  rouge_rouge2_13  rouge_rougeL_13
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L2    L2         0.727273         0.285714         0.590909         0.700000         0.315789         0.600000
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L3    L3              NaN              NaN              NaN              NaN              NaN              NaN
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L4    L4         0.206897         0.000000         0.137931         0.257143         0.000000         0.171429
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L2    L2         0.352941         0.000000         0.235294              NaN              NaN              NaN
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L3    L3         0.367816         0.117647 

In [27]:

# ============================================================
# Bottom 5 units per level by mean ROUGE-L (most discriminative)
# ============================================================
print("\n=== Lowest ROUGE-L Units per Level ===\n")

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()

    rougeL_cols = [ROUGE_PAIR_COLS[(f"{a.split('_')[-1]}{b.split('_')[-1]}", "rougeL")]
                   for (a, b) in ANNOTATOR_PAIRS]
    sub["mean_rougeL"] = sub[rougeL_cols].mean(axis=1, skipna=True)
    sub_valid = sub[sub["mean_rougeL"].notna()].sort_values("mean_rougeL")

    print(f"{'='*70}")
    print(f"LEVEL {level} — Bottom 5 by mean ROUGE-L")
    print(f"{'='*70}")

    for _, row in sub_valid.head(5).iterrows():
        print(f"\nunit : {row['unit_id']}")
        print(f"chart: {row['chart_id']} — {row['title']}")
        print(f"ROUGE-L: {row['mean_rougeL']:.4f}")
        print(f"ann1 : {row['annotator_1']}")
        print(f"ann2 : {row['annotator_2']}")
        print(f"ann3 : {row['annotator_3']}")


=== Lowest ROUGE-L Units per Level ===

LEVEL L2 — Bottom 5 by mean ROUGE-L

unit : 511bca34-e364-49d6-8cf9-263c3426db13__chart2__L2
chart: 2 — Profit Margin by State
ROUGE-L: 0.0800
ann1 : The majority of states show a positive margin of green encoded in 2021, compared to 2020.
ann2 : AK, WY, HI, and ME had N/A value.
ann3 : None

unit : dc09d834-178b-48e2-990d-94464475d0fa__chart5__L2
chart: 5 — Monthly Orders Details
ROUGE-L: 0.1174
ann1 : The highest orders occurred in November 2019 at around 250 orders.
ann2 : Ranging from 2016 to 2019, almost every month in every year, the number of the later year is more than the earlier.
ann3 : 2019 has the biggest monthly orders compared with 2018, 2017, and 2016.

unit : b164ac02-5f8b-4b10-ad35-afa018951888__chart2__L2
chart: 2 — 2023 | Sales vs Targets
ROUGE-L: 0.1536
ann1 : Sales started at around £40K in January and ended the year at around £80K, but it did not reach the target.
ann2 : In 2023, the sales vs target, J is 21.7K, F is -7.3K,

### BLEU Trial

In [28]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt')
nltk.download('punkt_tab')

smoother = SmoothingFunction().method1

def bleu_pairwise_batch(refs, hyps):
    assert len(refs) == len(hyps)
    results = {
        "bleu1": np.full(len(refs), np.nan),
        "bleu2": np.full(len(refs), np.nan),
        "bleu4": np.full(len(refs), np.nan),
    }
    for i, (r, h) in enumerate(zip(refs, hyps)):
        if is_missing(r) or is_missing(h):
            continue
        ref_tokens = nltk.word_tokenize(str(r).lower())
        hyp_tokens = nltk.word_tokenize(str(h).lower())
        results["bleu1"][i] = sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smoother
        )
        results["bleu2"][i] = sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smoother
        )
        results["bleu4"][i] = sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoother
        )
    return results

BLEU_METRICS = ["bleu1", "bleu2", "bleu4"]
BLEU_PAIR_COLS = {}

for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"{suffix_a}{suffix_b}"
    print(f"Computing BLEU for ann{suffix_a} vs ann{suffix_b} ...")
    bleu_results = bleu_pairwise_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist()
    )
    for metric in BLEU_METRICS:
        col_name = f"bleu_{metric}_{pair_key}"
        long_df[col_name] = bleu_results[metric]
        BLEU_PAIR_COLS[(pair_key, metric)] = col_name

print("\n=== Mean Pairwise BLEU by Level ===\n")
bleu_iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}
    for metric in BLEU_METRICS:
        pair_scores = []
        pair_details = []
        for (col_a, col_b) in ANNOTATOR_PAIRS:
            suffix_a = col_a.split("_")[-1]
            suffix_b = col_b.split("_")[-1]
            pair_key = f"{suffix_a}{suffix_b}"
            col_name = BLEU_PAIR_COLS[(pair_key, metric)]
            mean_score = sub[col_name].mean(skipna=True)
            n_valid    = sub[col_name].notna().sum()
            pair_scores.append(mean_score)
            pair_details.append(f"ann{suffix_a}-ann{suffix_b}={mean_score:.4f}(n={n_valid})")
            row[f"{metric}_ann{suffix_a}_ann{suffix_b}"] = round(mean_score, 4)
        row[f"{metric}_mean"] = round(np.nanmean(pair_scores), 4)
        print(f"  {level} {metric.upper():6s}: "
              f"{' | '.join(pair_details)} | mean={row[f'{metric}_mean']:.4f}")
    bleu_iaa_results.append(row)
    print()

bleu_iaa_df = pd.DataFrame(bleu_iaa_results)

print("\n=== BLEU Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}:")
    for metric in BLEU_METRICS:
        pair_cols = [BLEU_PAIR_COLS[(f"{a.split('_')[-1]}{b.split('_')[-1]}", metric)]
                     for (a, b) in ANNOTATOR_PAIRS]
        vals = sub[pair_cols].values.flatten()
        vals = vals[~np.isnan(vals)]
        print(f"    {metric.upper():6s}: min={vals.min():.4f}  "
              f"median={np.median(vals):.4f}  "
              f"max={vals.max():.4f}")
    print()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Computing BLEU for ann1 vs ann2 ...
Computing BLEU for ann1 vs ann3 ...
Computing BLEU for ann2 vs ann3 ...

=== Mean Pairwise BLEU by Level ===

  L2 BLEU1 : ann1-ann2=0.4118(n=49) | ann1-ann3=0.3391(n=45) | ann2-ann3=0.4101(n=45) | mean=0.3870
  L2 BLEU2 : ann1-ann2=0.2502(n=49) | ann1-ann3=0.1898(n=45) | ann2-ann3=0.2385(n=45) | mean=0.2262
  L2 BLEU4 : ann1-ann2=0.0976(n=49) | ann1-ann3=0.0734(n=45) | ann2-ann3=0.0932(n=45) | mean=0.0881

  L3 BLEU1 : ann1-ann2=0.2447(n=25) | ann1-ann3=0.2326(n=25) | ann2-ann3=0.2851(n=46) | mean=0.2541
  L3 BLEU2 : ann1-ann2=0.1048(n=25) | ann1-ann3=0.1134(n=25) | ann2-ann3=0.1535(n=46) | mean=0.1239
  L3 BLEU4 : ann1-ann2=0.0364(n=25) | ann1-ann3=0.0374(n=25) | ann2-ann3=0.0488(n=46) | mean=0.0409

  L4 BLEU1 : ann1-ann2=0.1968(n=49) | ann1-ann3=0.2031(n=48) | ann2-ann3=0.2570(n=48) | mean=0.2190
  L4 BLEU2 : ann1-ann2=0.0481(n=49) | ann1-ann3=0.0608(n=48) | ann2-ann3=0.1041(n=48) | mean=0.0710
  L4 BLEU4 : ann1-ann2=0.0138(n=49) | ann1-ann3=0.01

### METEOR Trial

In [29]:
from nltk.translate.meteor_score import meteor_score
nltk.download('wordnet')

def meteor_pairwise_batch(refs, hyps):
    assert len(refs) == len(hyps)
    results = {"meteor": np.full(len(refs), np.nan)}
    for i, (r, h) in enumerate(zip(refs, hyps)):
        if is_missing(r) or is_missing(h):
            continue
        ref_tokens = nltk.word_tokenize(str(r).lower())
        hyp_tokens = nltk.word_tokenize(str(h).lower())
        results["meteor"][i] = meteor_score([ref_tokens], hyp_tokens)
    return results

METEOR_PAIR_COLS = {}

for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"{suffix_a}{suffix_b}"
    print(f"Computing METEOR for ann{suffix_a} vs ann{suffix_b} ...")
    meteor_results = meteor_pairwise_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist()
    )
    col_name = f"meteor_{pair_key}"
    long_df[col_name] = meteor_results["meteor"]
    METEOR_PAIR_COLS[pair_key] = col_name

print("\n=== Mean Pairwise METEOR by Level ===\n")
meteor_iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}
    pair_scores = []
    pair_details = []
    for (col_a, col_b) in ANNOTATOR_PAIRS:
        suffix_a = col_a.split("_")[-1]
        suffix_b = col_b.split("_")[-1]
        pair_key = f"{suffix_a}{suffix_b}"
        col_name = METEOR_PAIR_COLS[pair_key]
        mean_score = sub[col_name].mean(skipna=True)
        n_valid    = sub[col_name].notna().sum()
        pair_scores.append(mean_score)
        pair_details.append(f"ann{suffix_a}-ann{suffix_b}={mean_score:.4f}(n={n_valid})")
        row[f"meteor_ann{suffix_a}_ann{suffix_b}"] = round(mean_score, 4)
    row["meteor_mean"] = round(np.nanmean(pair_scores), 4)
    print(f"  {level} METEOR: "
          f"{' | '.join(pair_details)} | mean={row['meteor_mean']:.4f}")
    meteor_iaa_results.append(row)

meteor_iaa_df = pd.DataFrame(meteor_iaa_results)

print("\n=== METEOR Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    meteor_cols = [METEOR_PAIR_COLS[f"{a.split('_')[-1]}{b.split('_')[-1]}"]
                   for (a, b) in ANNOTATOR_PAIRS]
    vals = sub[meteor_cols].values.flatten()
    vals = vals[~np.isnan(vals)]
    print(f"  {level} METEOR: min={vals.min():.4f}  "
          f"median={np.median(vals):.4f}  "
          f"max={vals.max():.4f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...


Computing METEOR for ann1 vs ann2 ...
Computing METEOR for ann1 vs ann3 ...
Computing METEOR for ann2 vs ann3 ...

=== Mean Pairwise METEOR by Level ===

  L2 METEOR: ann1-ann2=0.3872(n=49) | ann1-ann3=0.3176(n=45) | ann2-ann3=0.3907(n=45) | mean=0.3652
  L3 METEOR: ann1-ann2=0.1902(n=25) | ann1-ann3=0.2101(n=25) | ann2-ann3=0.2926(n=46) | mean=0.2310
  L4 METEOR: ann1-ann2=0.1587(n=49) | ann1-ann3=0.1873(n=48) | ann2-ann3=0.2474(n=48) | mean=0.1978

=== METEOR Distribution Summary per Level ===

  L2 METEOR: min=0.0560  median=0.3459  max=0.9915
  L3 METEOR: min=0.0565  median=0.2298  max=0.5293
  L4 METEOR: min=0.0522  median=0.1813  max=0.5598


In [30]:
print("=== Full IAA Metric Comparison (mean across annotator pairs) ===\n")
print(f"{'Level':6} {'BLEU-1':8} {'BLEU-2':8} {'BLEU-4':8} "
      f"{'METEOR':8} {'ROUGE-1':8} {'ROUGE-2':8} {'ROUGE-L':8} {'BERTScore':10}")
print("-" * 78)

for level in ["L2", "L3", "L4"]:
    b  = bleu_iaa_df[bleu_iaa_df["Level"] == level].iloc[0]
    m  = meteor_iaa_df[meteor_iaa_df["Level"] == level].iloc[0]
    r  = rouge_iaa_df[rouge_iaa_df["Level"] == level].iloc[0]
    bs = iaa_df[iaa_df["Level"] == level].iloc[0]
    print(f"{level:6} "
          f"{b['bleu1_mean']:8.4f} "
          f"{b['bleu2_mean']:8.4f} "
          f"{b['bleu4_mean']:8.4f} "
          f"{m['meteor_mean']:8.4f} "
          f"{r['rouge1_mean']:8.4f} "
          f"{r['rouge2_mean']:8.4f} "
          f"{r['rougeL_mean']:8.4f} "
          f"{bs['Mean_F1_overall']:10.4f}")

=== Full IAA Metric Comparison (mean across annotator pairs) ===

Level  BLEU-1   BLEU-2   BLEU-4   METEOR   ROUGE-1  ROUGE-2  ROUGE-L  BERTScore 
------------------------------------------------------------------------------
L2       0.3870   0.2262   0.0881   0.3652   0.4901   0.2042   0.3947     0.9070
L3       0.2541   0.1239   0.0409   0.2310   0.2858   0.0703   0.2073     0.8749
L4       0.2190   0.0710   0.0190   0.1978   0.2538   0.0344   0.1562     0.8716
